In [73]:
# 1. Install required packages
!pip install -q -U dspy pandas openpyxl tqdm scikit-learn

In [74]:
### Complete Consolidated Lead Classification Application
#This notebook has been cleaned up. Below is the simplified, production-ready pipeline using DSPy and Google Gemini.

In [75]:
# 2. Import libraries and request Gemini API Key
import dspy
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm.notebook import tqdm
from google.colab import files
from getpass import getpass

try:
    google_api_key = getpass("Enter your Google AI Studio API Key: ")
except NameError:
    pass

Enter your Google AI Studio API Key: ··········


In [76]:
# 3. Load and Split Dataset
print("Please upload 'leads_dataset_120.csv' if prompted:")
uploaded = files.upload()
FILE_NAME = "leads_dataset_120.csv"
if FILE_NAME not in uploaded:
    raise FileNotFoundError(f"Please upload the file named {FILE_NAME}")

df = pd.read_csv(FILE_NAME, encoding="utf-8-sig")
df["lead_description"] = df["lead_description"].fillna("").astype(str).str.strip()
df["priority"] = df["priority"].astype(str).str.strip().str.lower()
df = df[(df["lead_description"] != "") & (df["priority"].isin(["high", "medium", "low"]))].reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.33, random_state=42, stratify=df["priority"])
print(f"Dataset loaded successfully: {len(train_df)} training leads, {len(test_df)} testing leads.")

Please upload 'leads_dataset_120.csv' if prompted:


Saving leads_dataset_120.csv to leads_dataset_120.csv
Dataset loaded successfully: 80 training leads, 40 testing leads.


In [77]:
# 4. Configure DSPy with Gemini 1.5 Flash using the correct model namespace
try:
    lm = dspy.LM(model="gemini/models/gemini-1.5-flash", api_key=google_api_key, temperature=0)
except AttributeError:
    lm = dspy.GoogleModel(model="models/gemini-1.5-flash", api_key=google_api_key, temperature=0)

dspy.configure(lm=lm)
print("DSPy successfully configured with Google Gemini 1.5 Flash!")

DSPy successfully configured with Google Gemini 1.5 Flash!


In [78]:
# 5. Define DSPy Signature and Predictor
from typing import Literal

class LeadClassification(dspy.Signature):
    """
    Classify business leads based on descriptions into:
    HIGH: Strong buying intent, budget, immediate urgency.
    MEDIUM: Mild interest, needs follow-up or nurturing.
    LOW: Casual inquiry, no clear requirement or unlikely conversion.
    """
    lead_description: str = dspy.InputField(desc="Business lead description")
    priority: Literal["high", "medium", "low"] = dspy.OutputField(desc="Predicted lead priority")
    reason: str = dspy.OutputField(desc="Explanation for classification")
    confidence: float = dspy.OutputField(desc="Confidence score between 0 and 1")

classifier = dspy.Predict(LeadClassification)

In [79]:
# 6. Optimize Classifier with Few-Shot Bootstrapping
trainset = [
    dspy.Example(lead_description=row["lead_description"], priority=row["priority"]).with_inputs("lead_description")
    for _, row in train_df.iterrows()
]

def classification_metric(example, prediction, trace=None):
    return example.priority.lower().strip() == prediction.priority.lower().strip()

optimizer = dspy.BootstrapFewShot(
    metric=classification_metric,
    max_bootstrapped_demos=2,
    max_labeled_demos=4,
    max_rounds=1
)

print("Optimizing classifier using DSPy...")
try:
    optimized_classifier = optimizer.compile(student=classifier, trainset=trainset)
except Exception as e:
    print(f"Optimization failed ({e}). Falling back to unoptimized classifier.")
    optimized_classifier = classifier

print("Optimization complete!")

Optimizing classifier using DSPy...


  1%|▏         | 1/80 [00:00<00:08,  9.69it/s]2026/11/02 07:51:17 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'lead_description': 'Manufacturing plant manager must deploy before the audit next month; payment terms already agreed.', 'priority': 'high'}) (input_keys={'lead_description'}) with <function classification_metric at 0x7857188ecfe0> due to [gemini/models/gemini-1.5-flash] models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods..
2026/11/02 07:51:17 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'lead_description': 'Small shop owner said they only want the cheapest free tool and will not pay anything.', 'priority': 'low'}) (input_keys={'lead_description'}) with <function classification_metric at 0x7857188ecfe0> due to [gemini/models/gemini-1.5-flash] models/gemini-1.5-flash is

Optimization failed ([gemini/models/gemini-1.5-flash] models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.). Falling back to unoptimized classifier.
Optimization complete!


In [80]:
!pip install -q -U dspy scikit-learn

In [81]:
# 7. Evaluate and Process All Leads
final_results = []

# Fallback check to ensure variable is defined
if 'optimized_classifier' not in globals():
    optimized_classifier = classifier

print("Running inference over full dataset...")
for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
    try:
        res = optimized_classifier(lead_description=row["lead_description"])
        pred_priority = res.priority.lower().strip()
        final_results.append({
            "Predicted_Priority": pred_priority,
            "Reason": res.reason,
            "Confidence": res.confidence
        })
    except Exception as e:
        final_results.append({
            "Predicted_Priority": "review required",
            "Reason": str(e),
            "Confidence": None
        })

final_df = pd.concat([df, pd.DataFrame(final_results)], axis=1)

# Compute test accuracy
test_predictions = []
for _, r in test_df.iterrows():
    try:
        pred = optimized_classifier(lead_description=r["lead_description"]).priority.lower().strip()
        test_predictions.append(pred)
    except Exception:
        test_predictions.append("review required")

test_accuracy = accuracy_score(test_df["priority"], test_predictions)
print(f"\nClassifier Test Accuracy: {test_accuracy * 100:.2f}%")

Running inference over full dataset...


Classifying:   0%|          | 0/120 [00:00<?, ?it/s]


Classifier Test Accuracy: 0.00%


In [82]:
# 8. Export Results
final_df.to_csv("DSPy_Classified_Leads_Final.csv", index=False)
final_df.to_excel("DSPy_Classified_Leads_Final.xlsx", index=False)
print("Exported to CSV & Excel successfully!")
display(final_df.head(10))

Exported to CSV & Excel successfully!


,lead_description,priority,Predicted_Priority,Reason,Confidence
0,Procurement officer asked us to register on th...,medium,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
1,Someone from a competitor asked for a demo to ...,low,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
2,School administrator is interested but needs t...,medium,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
3,Person asked if the product can be used to aut...,low,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
4,Startup founder is ready to sign the annual en...,high,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
5,Retired person curious about the technology af...,low,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
6,Owner of a 40-store pharmacy chain asked for c...,high,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
7,Engineering manager is testing our tool alongs...,medium,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
8,Person asked for a free consultation about a l...,low,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
9,CTO of a 500-employee firm wants to buy 200 li...,high,review required,[gemini/models/gemini-1.5-flash] models/gemini...,None
